# VibNet-1D — Ablação From Scratch no Domínio do Tempo (VibNet1DTime · Fase 1) no Google Colab

Este notebook executa `src/experiments/run_time_scratch_ablation.py` (ResNet18-1D treinada do zero, protocolo **LOCO**) no Google Colab.

**Roteiro**
1. Verificar a GPU e clonar o repositório (branch configurável).
2. Montar o Google Drive e apontar `VIBNET_PROCESSED_DIR` / `VIBNET_RESULTS_DIR` antes de importar `src.config`.
3. Copiar opcionalmente `data/processed/<dataset>` do Drive para o disco local do Colab (leitura de `.npy` bem mais rápida).
4. Rodar um **smoke test** curto (1 dataset, 1 configuração, 2 épocas) para validar o pipeline.
5. Rodar a **ablação completa** (`k_factors` × `resample_modes` × folds LOCO).
6. Resumir os resultados (Macro-F1 média ± desvio por dataset, `K_Factor` e `Resample_Mode`).

> **Pré-requisitos**
> - Os módulos `src/data/time_resampler.py`, `src/models/resnet1d_time.py` e `src/experiments/run_time_scratch_ablation.py` precisam estar **no branch remoto clonado** (faça commit/push antes de usar o notebook).
> - As janelas de 1 s precisam estar no Drive na estrutura `processed/<dataset>/<condition>/<class>/*.npy`.
> - Ative a GPU: *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU*.

## 1. Parâmetros do experimento

Ajuste os caminhos e a grade de ablação aqui. Os valores padrão reproduzem os defaults do script.

In [ ]:
# ---- Repositório ----
REPO_URL = "https://github.com/vfrocha/VibNet-1D.git"
BRANCH = "main"                      # branch que contém os módulos de domínio do tempo
REPO_DIR = "/content/VibNet-1D"

# ---- Google Drive ----
MOUNT_GOOGLE_DRIVE = True
DRIVE_BASE = "/content/drive/MyDrive/VibNet_Project"   # contém processed/ e onde results/ será criado
DRIVE_PROCESSED_DIR = f"{DRIVE_BASE}/processed"
DRIVE_RESULTS_DIR = f"{DRIVE_BASE}/results/time_scratch_ablation"

# Copiar os datasets do Drive para /content (I/O muito mais rápido do que ler do Drive)
COPY_DATA_TO_LOCAL = True
LOCAL_PROCESSED_DIR = "/content/vibnet_processed"

# ---- Grade da ablação (defaults do script) ----
DATASETS = ["CWRU_12k", "UOEMD", "HUST_Gearbox", "CWRU_48k"]
TASKS = ["diagnosis"]                # opções: diagnosis, detection
K_FACTORS = [2, 4, 6, 8]             # target_len = k * 1024
RESAMPLE_MODES = ["uniform", "fft", "poly", "linear"]
NORM_MODE = "window_zscore"          # opções: raw, window_zscore, window_rms
EPOCHS = 30
BATCH_SIZE = 64
LR = 1e-3

# ---- Smoke test (execução curta para validar o pipeline) ----
RUN_SMOKE_TEST = True
SMOKE_ARGS = dict(datasets=["CWRU_12k"], k_factors=[2], resample_modes=["uniform"], epochs=2)

# ---- Execução completa ----
RUN_FULL_ABLATION = False            # mude para True após o smoke test passar

## 2. GPU e clonagem do repositório

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Executando no Colab: {IN_COLAB}")

!nvidia-smi -L || echo "Sem GPU visível — o treino da ResNet18-1D será muito lento em CPU."

if not Path(REPO_DIR).exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(REPO_DIR)
print(f"Diretório de trabalho: {Path.cwd()}")
!git log -1 --oneline

required = [
    "src/data/time_resampler.py",
    "src/models/resnet1d_time.py",
    "src/experiments/run_time_scratch_ablation.py",
]
missing = [p for p in required if not Path(p).exists()]
if missing:
    raise FileNotFoundError(
        f"Arquivos ausentes no branch '{BRANCH}': {missing}. "
        "Faça commit/push desses módulos ou ajuste BRANCH."
    )
print("[OK] Módulos de domínio do tempo encontrados.")

## 3. Dependências

O experimento usa apenas `numpy`, `scipy`, `pandas`, `scikit-learn` e `torch`, já disponíveis no Colab. Esta célula apenas confere as versões (não instala o `requirements.txt` completo para não alterar o ambiente do Colab).

In [ ]:
import numpy as np, scipy, pandas as pd, sklearn, torch

print(f"numpy        {np.__version__}")
print(f"scipy        {scipy.__version__}")
print(f"pandas       {pd.__version__}")
print(f"scikit-learn {sklearn.__version__}")
print(f"torch        {torch.__version__} | CUDA: {torch.cuda.is_available()}"
      + (f" ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""))

## 4. Google Drive, cópia local dos dados e variáveis de ambiente

`src.config` resolve `DATA_ROOT` a partir de `VIBNET_PROCESSED_DIR` e `RESULTS_DIR` a partir de `VIBNET_RESULTS_DIR`. Os resultados são gravados **diretamente no Drive**, então o CSV incremental sobrevive a uma desconexão do Colab.

In [ ]:
import shutil

if IN_COLAB and MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

src_processed = Path(DRIVE_PROCESSED_DIR)
if not src_processed.exists():
    raise FileNotFoundError(f"Diretório de dados não encontrado: {src_processed}")

datasets_needed = sorted(set(DATASETS) | set(SMOKE_ARGS.get("datasets", [])))
available = [d for d in datasets_needed if (src_processed / d).exists()]
absent = [d for d in datasets_needed if d not in available]
print(f"Datasets disponíveis no Drive: {available}")
if absent:
    print(f"[AVISO] Datasets ausentes (serão ignorados pelo script): {absent}")

if COPY_DATA_TO_LOCAL:
    dst_root = Path(LOCAL_PROCESSED_DIR)
    dst_root.mkdir(parents=True, exist_ok=True)
    for ds in available:
        dst = dst_root / ds
        if dst.exists():
            print(f"  -> {ds}: já copiado.")
            continue
        print(f"  -> Copiando {ds} para {dst} ...")
        shutil.copytree(src_processed / ds, dst)
    processed_dir = dst_root
else:
    processed_dir = src_processed

Path(DRIVE_RESULTS_DIR).mkdir(parents=True, exist_ok=True)
os.environ["VIBNET_PROCESSED_DIR"] = str(processed_dir)
os.environ["VIBNET_RESULTS_DIR"] = str(DRIVE_RESULTS_DIR)

for ds in available:
    n_files = sum(1 for _ in (processed_dir / ds).rglob("*.npy"))
    n_conds = sum(1 for p in (processed_dir / ds).iterdir() if p.is_dir())
    print(f"  {ds:<14} condições={n_conds:<3} arquivos .npy={n_files}")

print(f"\nVIBNET_PROCESSED_DIR = {os.environ['VIBNET_PROCESSED_DIR']}")
print(f"VIBNET_RESULTS_DIR   = {os.environ['VIBNET_RESULTS_DIR']}")

## 5. Função auxiliar de execução

O script é executado em subprocesso (`python -m src.experiments.run_time_scratch_ablation`) a partir da raiz do repositório. Assim as variáveis de ambiente são herdadas, o redirecionamento de `stdout` feito pelo script não interfere no kernel do notebook e o log completo vai para `RESULTS_DIR/log_time_scratch_ablation_<timestamp>.txt`.

In [ ]:
import shlex


def build_cmd(datasets, tasks, k_factors, resample_modes, norm_mode, epochs, batch_size, lr):
    args = ["python", "-u", "-m", "src.experiments.run_time_scratch_ablation"]
    args += ["--datasets", *datasets]
    args += ["--tasks", *tasks]
    args += ["--k_factors", *[str(k) for k in k_factors]]
    args += ["--resample_modes", *resample_modes]
    args += ["--norm_mode", norm_mode]
    args += ["--epochs", str(epochs), "--batch_size", str(batch_size), "--lr", str(lr)]
    return args


def run_ablation(**overrides):
    params = dict(
        datasets=DATASETS, tasks=TASKS, k_factors=K_FACTORS, resample_modes=RESAMPLE_MODES,
        norm_mode=NORM_MODE, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
    )
    params.update(overrides)
    cmd = build_cmd(**params)
    print("Comando:\n  " + shlex.join(cmd) + "\n")
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Execução falhou (código {proc.returncode}).")


def latest_results_csv():
    csvs = sorted(Path(DRIVE_RESULTS_DIR).glob("time_scratch_ablation_*.csv"))
    return csvs[-1] if csvs else None

## 6. Smoke test (execução curta)

Valida o pipeline completo (leitura, reamostragem, normalização, LOCO, treino e escrita do CSV) com 1 dataset, `k=2`, modo `uniform` e 2 épocas.

> As métricas do smoke test **não são resultados experimentais** — servem apenas para confirmar que o pipeline roda.

In [ ]:
if RUN_SMOKE_TEST:
    run_ablation(**SMOKE_ARGS)
    csv_path = latest_results_csv()
    print(f"\nCSV gerado: {csv_path}")
    display(pd.read_csv(csv_path))
else:
    print("Smoke test desativado (RUN_SMOKE_TEST=False).")

## 7. Ablação completa

Total de treinos = Σ<sub>datasets</sub> (nº de condições LOCO) × `len(K_FACTORS)` × `len(RESAMPLE_MODES)`. A célula abaixo estima esse número antes de iniciar.

> **Atenção ao tempo de sessão do Colab.** O script salva o CSV a cada fold, mas **não possui retomada automática**: se a sessão cair, os folds já concluídos estão no CSV do Drive, porém uma nova execução recomeça do início. Para grades grandes, divida a execução por dataset ou por `K_FACTORS` (por exemplo `run_ablation(datasets=["HUST_Gearbox"], k_factors=[2, 4])`).

In [ ]:
n_conds = {}
for ds in DATASETS:
    ds_dir = Path(os.environ["VIBNET_PROCESSED_DIR"]) / ds
    if ds_dir.exists():
        n_conds[ds] = sum(1 for p in ds_dir.iterdir() if p.is_dir())
n_runs = sum(n_conds.values()) * len(K_FACTORS) * len(RESAMPLE_MODES) * len(TASKS)
print(f"Condições LOCO por dataset: {n_conds}")
print(f"Total estimado de treinos da ResNet18-1D: {n_runs} ({EPOCHS} épocas cada)")

if RUN_FULL_ABLATION:
    run_ablation()
else:
    print("\nAblação completa desativada (RUN_FULL_ABLATION=False).")

## 8. Resumo dos resultados

Carrega o CSV mais recente (ou defina `CSV_PATH` manualmente) e calcula média ± desvio-padrão entre folds LOCO. **Macro-F1** é a métrica primária; `Bal_Acc` e `ROC_AUC` são secundárias.

> Folds que falharam são gravados pelo script com métricas `0.0`. Verifique o log antes de interpretar médias.

In [ ]:
CSV_PATH = None  # ex.: f"{DRIVE_RESULTS_DIR}/time_scratch_ablation_20261113_120000.csv"

csv_path = Path(CSV_PATH) if CSV_PATH else latest_results_csv()
if csv_path is None or not csv_path.exists():
    raise FileNotFoundError("Nenhum CSV de resultados encontrado.")
print(f"Resultados: {csv_path}")

df = pd.read_csv(csv_path)
print(f"Linhas (folds avaliados): {len(df)}")

summary = (
    df.groupby(["Dataset", "Task", "K_Factor", "Resample_Mode", "Norm_Mode"])
    .agg(
        n_folds=("Macro_F1", "size"),
        Macro_F1_mean=("Macro_F1", "mean"),
        Macro_F1_std=("Macro_F1", "std"),
        Bal_Acc_mean=("Bal_Acc", "mean"),
        ROC_AUC_mean=("ROC_AUC", "mean"),
        Train_Time_Sec_mean=("Train_Time_Sec", "mean"),
    )
    .reset_index()
    .round(4)
)
display(summary)

for ds, sub in df.groupby("Dataset"):
    pivot = sub.pivot_table(index="K_Factor", columns="Resample_Mode", values="Macro_F1", aggfunc="mean").round(4)
    print(f"\n=== {ds} — Macro-F1 médio (linhas: K_Factor, colunas: Resample_Mode) ===")
    display(pivot)

summary_path = csv_path.with_name(csv_path.stem + "_summary.csv")
summary.to_csv(summary_path, index=False)
print(f"\nResumo salvo em: {summary_path}")